In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder

E:\CondaEnvs\smartagrivision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cleaned Dataset Paths

In [2]:


# CLEANED DATASET ROOT

CLEANED_DATASET_ROOT = Path("Cleaned_Dataset")

print("CLEANED DATASET PATHS")

print(
    f"Cleaned Dataset Root : "
    f"{CLEANED_DATASET_ROOT.resolve()}"
)

print(
    f"Exists               : "
    f"{CLEANED_DATASET_ROOT.exists()}"
)

if not CLEANED_DATASET_ROOT.exists():
    raise FileNotFoundError(
        "Cleaned_Dataset folder not found. "
        "Please run 02_Data_Cleaning.ipynb first."
    )


# DATASET FOLDERS

datasets = {
    "Fruits-360":
        CLEANED_DATASET_ROOT / "Fruits-360",

    "New Plant Diseases":
        CLEANED_DATASET_ROOT / "New Plant Diseases",

    "PlantDoc":
        CLEANED_DATASET_ROOT / "PlantDoc",

    "Vegetable Dataset":
        CLEANED_DATASET_ROOT / "Vegetable Dataset"
}


# CHECK DATASET AVAILABILITY

print()

for dataset_name, dataset_path in datasets.items():

    print(
        f"{dataset_name:<25} : "
        f"{'Available' if dataset_path.exists() else 'Missing'}"
    )

CLEANED DATASET PATHS
Cleaned Dataset Root : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Cleaned_Dataset
Exists               : True

Fruits-360                : Available
New Plant Diseases        : Available
PlantDoc                  : Available
Vegetable Dataset         : Available


# Dataset Structure Verification

In [4]:
# Cleaned Dataset Mapping

print("SECTION 3.1 : CLEANED DATASET PATH MAPPING")


cleaned_datasets = {
    "Fruits-360":
        CLEANED_DATASET_ROOT / "Fruits-360",

    "New Plant Diseases":
        CLEANED_DATASET_ROOT / "New Plant Diseases",

    "PlantDoc":
        CLEANED_DATASET_ROOT / "PlantDoc",

    "Vegetable Dataset":
        CLEANED_DATASET_ROOT / "Vegetable Dataset"
}


for dataset_name, dataset_path in cleaned_datasets.items():

    print()
    print(f"{dataset_name}")
    print(f"PATH   : {dataset_path}")
    print(f"EXISTS : {dataset_path.exists()}")

print()

SECTION 3.1 : CLEANED DATASET PATH MAPPING

Fruits-360
PATH   : Cleaned_Dataset\Fruits-360
EXISTS : True

New Plant Diseases
PATH   : Cleaned_Dataset\New Plant Diseases
EXISTS : True

PlantDoc
PATH   : Cleaned_Dataset\PlantDoc
EXISTS : True

Vegetable Dataset
PATH   : Cleaned_Dataset\Vegetable Dataset
EXISTS : True



In [5]:
# Detect Train / Test / Validation

print("TRAIN / TEST / VALIDATION STRUCTURE")



def find_split_folder(dataset_path, possible_names):

    for folder in dataset_path.iterdir():

        if not folder.is_dir():
            continue

        folder_name = (
            folder.name
            .strip()
            .lower()
        )

        if folder_name in possible_names:
            return folder

    return None


structure_records = []


for dataset_name, dataset_path in cleaned_datasets.items():

    print()
    print(f"DATASET : {dataset_name}")

    if not dataset_path.exists():

        print("STATUS  : MISSING")

        structure_records.append({
            "Dataset": dataset_name,
            "Train": "Missing",
            "Validation": "Missing",
            "Test": "Missing"
        })

        continue


    train_path = find_split_folder(
        dataset_path,
        {"train", "training"}
    )

    validation_path = find_split_folder(
        dataset_path,
        {
            "valid",
            "validation",
            "val"
        }
    )

    test_path = find_split_folder(
        dataset_path,
        {"test", "testing"}
    )


    print(
        f"Train      : "
        f"{train_path.name if train_path else 'Not Found'}"
    )

    print(
        f"Validation : "
        f"{validation_path.name if validation_path else 'Not Found'}"
    )

    print(
        f"Test       : "
        f"{test_path.name if test_path else 'Not Found'}"
    )


    structure_records.append({

        "Dataset": dataset_name,

        "Train":
            train_path.name
            if train_path
            else "Not Found",

        "Validation":
            validation_path.name
            if validation_path
            else "Not Found",

        "Test":
            test_path.name
            if test_path
            else "Not Found"

    })


structure_df = pd.DataFrame(
    structure_records
)


print()

print("SPLIT STRUCTURE SUMMARY")


display(structure_df)

print("=" * 90)

TRAIN / TEST / VALIDATION STRUCTURE

DATASET : Fruits-360
Train      : Not Found
Validation : Not Found
Test       : Not Found

DATASET : New Plant Diseases
Train      : train
Validation : valid
Test       : Not Found

DATASET : PlantDoc
Train      : train
Validation : Not Found
Test       : test

DATASET : Vegetable Dataset
Train      : train
Validation : validation
Test       : test

SPLIT STRUCTURE SUMMARY


,Dataset,Train,Validation,Test
0,Fruits-360,Not Found,Not Found,Not Found
1,New Plant Diseases,train,valid,Not Found
2,PlantDoc,train,Not Found,test
3,Vegetable Dataset,train,validation,test


In [11]:

 # CLASS & IMAGE STRUCTURE VERIFICATION



print("CLASS & IMAGE STRUCTURE VERIFICATION")



def count_images(folder):

    if folder is None:
        return 0

    if not folder.exists():
        return 0

    total = 0

    for path in folder.rglob("*"):

        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            total += 1

    return total


class_structure_records = []


for dataset_name, dataset_path in cleaned_datasets.items():

    if not dataset_path.exists():
        continue

    print()
    print(f"DATASET : {dataset_name}")

    total_classes = 0
    total_images = 0

    # Find Train / Validation / Test folders
    split_folders = {
        "Train": find_split_folder(
            dataset_path,
            {"train", "training"}
        ),

        "Validation": find_split_folder(
            dataset_path,
            {
                "valid",
                "validation",
                "val"
            }
        ),

        "Test": find_split_folder(
            dataset_path,
            {"test", "testing"}
        )
    }

    # Count classes and images
    for split_name, split_path in split_folders.items():

        if split_path is None:
            print(
                f"{split_name:<12} "
                f"Not Found"
            )
            continue

        class_folders = [
            folder
            for folder in split_path.iterdir()
            if folder.is_dir()
        ]

        class_count = len(class_folders)

        image_count = count_images(
            split_path
        )

        total_classes = max(
            total_classes,
            class_count
        )

        total_images += image_count

        print(
            f"{split_name:<12} "
            f"Classes : {class_count:,} | "
            f"Images : {image_count:,}"
        )

    class_structure_records.append({
        "Dataset": dataset_name,
        "Classes": total_classes,
        "Total Images": total_images
    })


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================

class_structure_df = pd.DataFrame(
    class_structure_records
)


print()

print("CLASS & IMAGE STRUCTURE SUMMARY")


display(class_structure_df)


print("SECTION 3 COMPLETED")


CLASS & IMAGE STRUCTURE VERIFICATION

DATASET : Fruits-360
Train        Not Found
Validation   Not Found
Test         Not Found

DATASET : New Plant Diseases
Train        Classes : 38 | Images : 70,277
Validation   Classes : 38 | Images : 17,564
Test         Not Found

DATASET : PlantDoc
Train        Classes : 28 | Images : 2,664
Validation   Not Found
Test         Classes : 27 | Images : 252

DATASET : Vegetable Dataset
Train        Classes : 15 | Images : 14,996
Validation   Classes : 15 | Images : 3,000
Test         Classes : 15 | Images : 3,000

CLASS & IMAGE STRUCTURE SUMMARY


,Dataset,Classes,Total Images
0,Fruits-360,0,0
1,New Plant Diseases,38,87841
2,PlantDoc,28,2916
3,Vegetable Dataset,15,20996


SECTION 3 COMPLETED


# Image Loading & Label Extraction

In [15]:
# IMAGE LOADING & LABEL EXTRACTION
print("IMAGE LOADING & LABEL EXTRACTION")



image_records = []


for dataset_name, dataset_path in cleaned_datasets.items():

    if not dataset_path.exists():

        print(
            f"Skipping : {dataset_name} "
            f"(Path not found)"
        )

        continue


    print()
    print(f"Scanning : {dataset_name}")


    image_paths = [

        path

        for path in dataset_path.rglob("*")

        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        )

    ]


    print(
        f"Images Found : "
        f"{len(image_paths):,}"
    )


    for image_path in tqdm(

        image_paths,

        desc=f"Extracting {dataset_name}",

        unit="image"

    ):

        try:

            relative_parts = (
                image_path
                .relative_to(dataset_path)
                .parts
            )


            split = "Unknown"
            class_name = "Unknown"


            # Detect Train / Test / Validation
          

            if len(relative_parts) >= 3:

                first_folder = (
                    relative_parts[0]
                    .lower()
                    .strip()
                )


                if first_folder in {
                    "train",
                    "training"
                }:

                    split = "Train"
                    class_name = relative_parts[1]


                elif first_folder in {
                    "test",
                    "testing"
                }:

                    split = "Test"
                    class_name = relative_parts[1]


                elif first_folder in {
                    "valid",
                    "validation",
                    "val"
                }:

                    split = "Validation"
                    class_name = relative_parts[1]


                else:

                    # Dataset/Class/Image
                    split = "Unsplit"
                    class_name = relative_parts[0]


            elif len(relative_parts) >= 2:

                class_name = relative_parts[0]


            image_records.append({

                "Dataset":
                    dataset_name,

                "Split":
                    split,

                "Class":
                    class_name,

                "Image_Path":
                    str(image_path),

                "Filename":
                    image_path.name

            })


        except Exception:

            continue


IMAGE LOADING & LABEL EXTRACTION

Scanning : Fruits-360
Images Found : 182,942


Extracting Fruits-360: 100%|██████████| 182942/182942 [00:02<00:00, 79655.33image/s]



Scanning : New Plant Diseases
Images Found : 87,841


Extracting New Plant Diseases: 100%|██████████| 87841/87841 [00:01<00:00, 75171.64image/s]



Scanning : PlantDoc
Images Found : 2,916


Extracting PlantDoc: 100%|██████████| 2916/2916 [00:00<00:00, 51278.29image/s]


Scanning : Vegetable Dataset


Images Found : 20,996


Extracting Vegetable Dataset: 100%|██████████| 20996/20996 [00:00<00:00, 72943.69image/s]


In [16]:
# Create DataFrame


image_inventory_df = pd.DataFrame(
    image_records
)


print()

print("IMAGE LOADING & LABEL EXTRACTION COMPLETED")


print(
    f"Total Image Records : "
    f"{len(image_inventory_df):,}"
)

if not image_inventory_df.empty:

    print(
        f"Total Datasets      : "
        f"{image_inventory_df['Dataset'].nunique():,}"
    )

    print(
        f"Total Classes       : "
        f"{image_inventory_df['Class'].nunique():,}"
    )

print()

display(
    image_inventory_df.head(10)
)



IMAGE LOADING & LABEL EXTRACTION COMPLETED
Total Image Records : 294,695
Total Datasets      : 4
Total Classes       : 82



,Dataset,Split,Class,Image_Path,Filename
0,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_103_100.jpg
1,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_107_100.jpg
2,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_111_100.jpg
3,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_115_100.jpg
4,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_119_100.jpg
5,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_11_100.jpg
6,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_123_100.jpg
7,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_127_100.jpg
8,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_131_100.jpg
9,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_135_100.jpg


# Image Resize

In [17]:

print("IMAGE RESIZE")
IMAGE_SIZE = (224, 224)


print(
    f"Target Image Size : "
    f"{IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}"
)


def resize_image(
    image_path,
    target_size=IMAGE_SIZE
):

    """
    Load an image and resize it.

    Returns:
        PIL Image in RGB format
    """

    with Image.open(image_path) as img:

        img = img.convert("RGB")

        resized_img = img.resize(
            target_size,
            Image.Resampling.LANCZOS
        )

        return resized_img


IMAGE RESIZE
Target Image Size : 224 x 224


In [18]:
# Test Resize

if not image_inventory_df.empty:

    sample_path = image_inventory_df.iloc[0][
        "Image_Path"
    ]

    try:

        sample_image = resize_image(
            sample_path
        )

        print()
        print("Resize Test : SUCCESS")

        print(
            f"Image Path : {sample_path}"
        )

        print(
            f"New Size   : {sample_image.size}"
        )

        print(
            f"Mode       : {sample_image.mode}"
        )


    except Exception as e:

        print()
        print("Resize Test : FAILED")
        print(f"Error : {e}")


else:

    print(
        "No image records available "
        "for resize test."
    )


print()





Resize Test : SUCCESS
Image Path : Cleaned_Dataset\Fruits-360\fruits-360_100x100\fruits-360\Test\Almonds 1\r0_103_100.jpg
New Size   : (224, 224)
Mode       : RGB



# Pixel Normalization

In [20]:
# SECTION 6 : PIXEL NORMALIZATION



print("PIXEL NORMALIZATION")


def normalize_image(image):

    """
    Convert image pixels from 0-255
    to 0-1.
    """

    image_array = np.asarray(
        image,
        dtype=np.float32
    )

    normalized = (
        image_array / 255.0
    )

    return normalized



PIXEL NORMALIZATION


In [21]:
# Test Normalization


if not image_inventory_df.empty:

    sample_path = image_inventory_df.iloc[0][
        "Image_Path"
    ]

    try:

        sample_image = resize_image(
            sample_path
        )

        normalized_image = normalize_image(
            sample_image
        )

        print()
        print("Normalization Test : SUCCESS")

        print(
            f"Shape       : "
            f"{normalized_image.shape}"
        )

        print(
            f"Data Type   : "
            f"{normalized_image.dtype}"
        )

        print(
            f"Minimum     : "
            f"{normalized_image.min():.4f}"
        )

        print(
            f"Maximum     : "
            f"{normalized_image.max():.4f}"
        )


    except Exception as e:

        print()
        print("Normalization Test : FAILED")
        print(f"Error : {e}")


print()



Normalization Test : SUCCESS
Shape       : (224, 224, 3)
Data Type   : float32
Minimum     : 0.0000
Maximum     : 1.0000



# Label Encoding

In [22]:

print("LABEL ENCODING")



label_encoder = LabelEncoder()


if image_inventory_df.empty:

    print(
        "No image records available "
        "for label encoding."
    )

else:

    # Remove unknown / blank class names
    valid_class_mask = (

        image_inventory_df["Class"]
        .astype(str)
        .str.strip()
        .ne("")

        &

        image_inventory_df["Class"]
        .astype(str)
        .ne("Unknown")

    )


    image_inventory_df = (
        image_inventory_df[
            valid_class_mask
        ]
        .copy()
    )


    # Fit encoder
    label_encoder.fit(
        image_inventory_df["Class"]
    )


    # Numerical labels
    image_inventory_df["Label"] = (
        label_encoder.transform(
            image_inventory_df["Class"]
        )
    )


    # Class mapping
    label_mapping_df = pd.DataFrame({

        "Label":
            range(
                len(label_encoder.classes_)
            ),

        "Class":
            label_encoder.classes_

    })


    print(
        f"Total Encoded Classes : "
        f"{len(label_encoder.classes_):,}"
    )


    print()
    print("Sample Label Mapping:")

    display(
        label_mapping_df.head(20)
    )


    print()
    print("Sample Image Records:")

    display(
        image_inventory_df[
            [
                "Dataset",
                "Split",
                "Class",
                "Label",
                "Image_Path"
            ]
        ].head(10)
    )


print()



LABEL ENCODING
Total Encoded Classes : 82

Sample Label Mapping:


,Label,Class
0,0,Apple_Scab_Leaf
1,1,Apple___Apple_scab
2,2,Apple___Black_rot
3,3,Apple___Cedar_apple_rust
4,4,Apple___healthy
5,5,Apple_leaf
6,6,Apple_rust_leaf
7,7,Bean
8,8,Bell_pepper_leaf
9,9,Bell_pepper_leaf_spot



Sample Image Records:


,Dataset,Split,Class,Label,Image_Path
0,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
1,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
2,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
3,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
4,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
5,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
6,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
7,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
8,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...
9,Fruits-360,Unsplit,fruits-360_100x100,79,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...


# Train / Validation / Test Preparation

In [27]:

# TRAIN / VALIDATION / TEST PREPARATION



print("TRAIN / VALIDATION / TEST PREPARATION")



if image_inventory_df.empty:

    print(
        "No image records available "
        "for split preparation."
    )

else:

    
    # Normalize split names


    split_map = {

        "train":
            "Train",

        "training":
            "Train",

        "test":
            "Test",

        "testing":
            "Test",

        "valid":
            "Validation",

        "validation":
            "Validation",

        "val":
            "Validation"

    }


    image_inventory_df["Split"] = (

        image_inventory_df["Split"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(split_map)
        .fillna(
            image_inventory_df["Split"]
        )

    )


   
    # Create Split DataFrames
    

    train_df = (
        image_inventory_df[
            image_inventory_df["Split"]
            == "Train"
        ]
        .copy()
        .reset_index(drop=True)
    )


    validation_df = (
        image_inventory_df[
            image_inventory_df["Split"]
            == "Validation"
        ]
        .copy()
        .reset_index(drop=True)
    )


    test_df = (
        image_inventory_df[
            image_inventory_df["Split"]
            == "Test"
        ]
        .copy()
        .reset_index(drop=True)
    )


   
    # Summary
   

    split_summary = pd.DataFrame({

        "Split": [
            "Train",
            "Validation",
            "Test"
        ],

        "Images": [
            len(train_df),
            len(validation_df),
            len(test_df)
        ]

    })


    print()
    
    print("TRAIN / VALIDATION / TEST SUMMARY")
   

    display(
        split_summary
    )


  
    # Dataset-wise Split Summary
  

    dataset_split_summary = (

        image_inventory_df
        .groupby(
            ["Dataset", "Split"]
        )
        .size()
        .reset_index(
            name="Images"
        )
        .sort_values(
            ["Dataset", "Split"]
        )

    )


    print()
    print("=" * 90)
    print("DATASET-WISE SPLIT SUMMARY")
    print("=" * 90)

    display(
        dataset_split_summary
    )


    
    # Check Unknown Splits
   

    unknown_split_df = (

        image_inventory_df[
            ~image_inventory_df["Split"].isin(
                [
                    "Train",
                    "Validation",
                    "Test"
                ]
            )
        ]

        .copy()

    )


    print()
    print(
        f"Unknown / Unsplit Images : "
        f"{len(unknown_split_df):,}"
    )


print()


TRAIN / VALIDATION / TEST PREPARATION

TRAIN / VALIDATION / TEST SUMMARY


,Split,Images
0,Train,87937
1,Validation,20564
2,Test,3252



DATASET-WISE SPLIT SUMMARY


,Dataset,Split,Images
0,Fruits-360,Unsplit,182942
1,New Plant Diseases,Train,70277
2,New Plant Diseases,Validation,17564
3,PlantDoc,Test,252
4,PlantDoc,Train,2664
5,Vegetable Dataset,Test,3000
6,Vegetable Dataset,Train,14996
7,Vegetable Dataset,Validation,3000



Unknown / Unsplit Images : 182,942



# Preprocessing Validation

In [28]:
#  PREPROCESSING VALIDATION - BASIC CHECK



print(" PREPROCESSING VALIDATION")


print(
    f"Total Image Records : "
    f"{len(image_inventory_df):,}"
)

print(
    f"Total Classes       : "
    f"{image_inventory_df['Class'].nunique():,}"
)

print(
    f"Total Datasets      : "
    f"{image_inventory_df['Dataset'].nunique():,}"
)

print(
    f"Image Size          : "
    f"{IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}"
)

print()

print("Required Columns:")

required_columns = [
    "Dataset",
    "Split",
    "Class",
    "Label",
    "Image_Path"
]

for column in required_columns:

    print(
        f"{column:<15} : "
        f"{'OK' if column in image_inventory_df.columns else 'MISSING'}"
    )


 PREPROCESSING VALIDATION
Total Image Records : 294,695
Total Classes       : 82
Total Datasets      : 4
Image Size          : 224 x 224

Required Columns:
Dataset         : OK
Split           : OK
Class           : OK
Label           : OK
Image_Path      : OK


In [29]:
# IMAGE PREPROCESSING TEST



print("IMAGE PREPROCESSING TEST")


if len(image_inventory_df) == 0:

    print("No image records available.")

else:

    sample_path = image_inventory_df.iloc[0]["Image_Path"]

    print(f"Sample Image : {sample_path}")

    try:

        # Load + resize
        processed_image = resize_image(
            sample_path
        )

        # Normalize
        normalized_image = normalize_image(
            processed_image
        )

        print()
        print("Image Loading     : PASS")
        print("Resize            : PASS")
        print("RGB Conversion    : PASS")
        print("Normalization     : PASS")

        print()
        print(
            f"Processed Shape   : "
            f"{normalized_image.shape}"
        )

        print(
            f"Data Type         : "
            f"{normalized_image.dtype}"
        )

        print(
            f"Pixel Min         : "
            f"{normalized_image.min():.4f}"
        )

        print(
            f"Pixel Max         : "
            f"{normalized_image.max():.4f}"
        )

    except Exception as e:

        print()
        print("PREPROCESSING TEST : FAILED")
        print(f"Error : {e}")


IMAGE PREPROCESSING TEST
Sample Image : Cleaned_Dataset\Fruits-360\fruits-360_100x100\fruits-360\Test\Almonds 1\r0_103_100.jpg

Image Loading     : PASS
Resize            : PASS
RGB Conversion    : PASS
Normalization     : PASS

Processed Shape   : (224, 224, 3)
Data Type         : float32
Pixel Min         : 0.0000
Pixel Max         : 1.0000


In [30]:
# Split Validation

print("SPLIT VALIDATION")


split_validation = pd.DataFrame({

    "Split": [
        "Train",
        "Validation",
        "Test"
    ],

    "Images": [
        len(train_df),
        len(validation_df),
        len(test_df)
    ]

})

split_validation["Percentage"] = (
    split_validation["Images"]
    / split_validation["Images"].sum()
    * 100
)

display(split_validation)

print(
    f"Total Split Images : "
    f"{split_validation['Images'].sum():,}"
)


SPLIT VALIDATION


,Split,Images,Percentage
0,Train,87937,78.688715
1,Validation,20564,18.401296
2,Test,3252,2.909989


Total Split Images : 111,753


# Preprocessed Dataset Summary

In [31]:
#  PREPROCESSED DATASET SUMMARY



print("PREPROCESSED DATASET SUMMARY")


preprocessed_summary = pd.DataFrame({

    "Metric": [
        "Total Images",
        "Total Classes",
        "Total Datasets",
        "Train Images",
        "Validation Images",
        "Test Images",
        "Image Width",
        "Image Height",
        "Pixel Range"
    ],

    "Value": [
        len(image_inventory_df),
        image_inventory_df["Class"].nunique(),
        image_inventory_df["Dataset"].nunique(),
        len(train_df),
        len(validation_df),
        len(test_df),
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        "0.0 - 1.0"
    ]

})

display(preprocessed_summary)


PREPROCESSED DATASET SUMMARY


,Metric,Value
0,Total Images,294695
1,Total Classes,82
2,Total Datasets,4
3,Train Images,87937
4,Validation Images,20564
5,Test Images,3252
6,Image Width,224
7,Image Height,224
8,Pixel Range,0.0 - 1.0


In [32]:
# DATASET-WISE PREPROCESSED SUMMARY

dataset_preprocessed_summary = (

    image_inventory_df

    .groupby("Dataset")

    .agg(

        Total_Images=(
            "Image_Path",
            "count"
        ),

        Classes=(
            "Class",
            "nunique"
        )

    )

    .reset_index()

)

display(
    dataset_preprocessed_summary
)

print()
print(
    f"Total Images : "
    f"{dataset_preprocessed_summary['Total_Images'].sum():,}"
)

print(
    f"Total Classes : "
    f"{dataset_preprocessed_summary['Classes'].sum():,}"
)

,Dataset,Total_Images,Classes
0,Fruits-360,182942,1
1,New Plant Diseases,87841,38
2,PlantDoc,2916,28
3,Vegetable Dataset,20996,15



Total Images : 294,695
Total Classes : 82


In [33]:
# DATASET-WISE SPLIT SUMMARY

dataset_split_summary = (

    image_inventory_df

    .groupby(
        [
            "Dataset",
            "Split"
        ]
    )

    .size()

    .reset_index(
        name="Images"
    )

    .sort_values(
        [
            "Dataset",
            "Split"
        ]
    )

)

display(
    dataset_split_summary
)


,Dataset,Split,Images
0,Fruits-360,Unsplit,182942
1,New Plant Diseases,Train,70277
2,New Plant Diseases,Validation,17564
3,PlantDoc,Test,252
4,PlantDoc,Train,2664
5,Vegetable Dataset,Test,3000
6,Vegetable Dataset,Train,14996
7,Vegetable Dataset,Validation,3000


# Save Preprocessed Data

In [34]:
# SECTION 11.1 : CREATE PREPROCESSED DATA DIRECTORY

print("CREATE PREPROCESSED DATA DIRECTORY")


PREPROCESSED_ROOT = (
    CLEANED_DATASET_ROOT.parent
    / "Preprocessed_Data"
)

PREPROCESSED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Preprocessed Data Path : "
    f"{PREPROCESSED_ROOT.resolve()}"
)

print(
    f"Exists                  : "
    f"{PREPROCESSED_ROOT.exists()}"
)

CREATE PREPROCESSED DATA DIRECTORY
Preprocessed Data Path : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Data
Exists                  : True


In [35]:
# SECTION 11.2 : SAVE PREPROCESSED METADATA

print("SAVE PREPROCESSED METADATA")


metadata_path = (
    PREPROCESSED_ROOT
    / "image_metadata.csv"
)

image_inventory_df.to_csv(
    metadata_path,
    index=False
)

print(
    f"Metadata Saved : "
    f"{metadata_path}"
)

print(
    f"Records Saved  : "
    f"{len(image_inventory_df):,}"
)

print(
    f"File Exists    : "
    f"{metadata_path.exists()}"
)


SAVE PREPROCESSED METADATA
Metadata Saved : Preprocessed_Data\image_metadata.csv
Records Saved  : 294,695
File Exists    : True


In [36]:
#  SAVE SPLIT METADATA

print("SAVE TRAIN / VALIDATION / TEST DATA")



train_path = (
    PREPROCESSED_ROOT
    / "train_metadata.csv"
)

validation_path = (
    PREPROCESSED_ROOT
    / "validation_metadata.csv"
)

test_path = (
    PREPROCESSED_ROOT
    / "test_metadata.csv"
)


train_df.to_csv(
    train_path,
    index=False
)

validation_df.to_csv(
    validation_path,
    index=False
)

test_df.to_csv(
    test_path,
    index=False
)


print(
    f"Train Metadata      : "
    f"{len(train_df):,}"
)

print(
    f"Validation Metadata : "
    f"{len(validation_df):,}"
)

print(
    f"Test Metadata       : "
    f"{len(test_df):,}"
)

print()
print("Files Saved:")
print(f"  ├── {train_path.name}")
print(f"  ├── {validation_path.name}")
print(f"  └── {test_path.name}")


SAVE TRAIN / VALIDATION / TEST DATA
Train Metadata      : 87,937
Validation Metadata : 20,564
Test Metadata       : 3,252

Files Saved:
  ├── train_metadata.csv
  ├── validation_metadata.csv
  └── test_metadata.csv


# Final Preprocessing Report

In [37]:
# SAVED FILE VALIDATION

print("SAVED FILE VALIDATION")


saved_files = [
    metadata_path,
    train_path,
    validation_path,
    test_path
]

for file_path in saved_files:

    print(
        f"{file_path.name:<30} "
        f": "
        f"{'OK' if file_path.exists() else 'MISSING'}"
    )


SAVED FILE VALIDATION
image_metadata.csv             : OK
train_metadata.csv             : OK
validation_metadata.csv        : OK
test_metadata.csv              : OK


In [38]:
# FINAL PREPROCESSING COUNTS

print("FINAL PREPROCESSING COUNTS")


total_images = len(image_inventory_df)

train_images = len(train_df)

validation_images = len(validation_df)

test_images = len(test_df)

total_classes = (
    image_inventory_df["Class"]
    .nunique()
)

total_datasets = (
    image_inventory_df["Dataset"]
    .nunique()
)


print(
    f"Total Images        : "
    f"{total_images:,}"
)

print(
    f"Train Images        : "
    f"{train_images:,}"
)

print(
    f"Validation Images   : "
    f"{validation_images:,}"
)

print(
    f"Test Images         : "
    f"{test_images:,}"
)

print(
    f"Total Classes       : "
    f"{total_classes:,}"
)

print(
    f"Total Datasets      : "
    f"{total_datasets:,}"
)

print(
    f"Image Size          : "
    f"{IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}"
)

print(
    "Pixel Normalization : 0.0 - 1.0"
)


FINAL PREPROCESSING COUNTS
Total Images        : 294,695
Train Images        : 87,937
Validation Images   : 20,564
Test Images         : 3,252
Total Classes       : 82
Total Datasets      : 4
Image Size          : 224 x 224
Pixel Normalization : 0.0 - 1.0


In [39]:
# FINAL PREPROCESSING REPORT

print("FINAL PREPROCESSING REPORT")



final_preprocessing_report = pd.DataFrame({

    "Metric": [

        "Total Images",
        "Train Images",
        "Validation Images",
        "Test Images",
        "Total Classes",
        "Total Datasets",
        "Image Width",
        "Image Height",
        "Pixel Minimum",
        "Pixel Maximum",
        "Metadata Saved"

    ],

    "Value": [

        total_images,
        train_images,
        validation_images,
        test_images,
        total_classes,
        total_datasets,
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        0.0,
        1.0,
        "Yes"

    ]

})


display(
    final_preprocessing_report
)


FINAL PREPROCESSING REPORT


,Metric,Value
0,Total Images,294695
1,Train Images,87937
2,Validation Images,20564
3,Test Images,3252
4,Total Classes,82
5,Total Datasets,4
6,Image Width,224
7,Image Height,224
8,Pixel Minimum,0.0
9,Pixel Maximum,1.0


In [40]:
# PREPROCESSING COMPLETED

print()

print()
print(
    "PREPROCESSING SUMMARY"
)

print(
    f"Total Images      : "
    f"{total_images:,}"
)

print(
    f"Total Classes     : "
    f"{total_classes:,}"
)

print(
    f"Train             : "
    f"{train_images:,}"
)

print(
    f"Validation        : "
    f"{validation_images:,}"
)

print(
    f"Test              : "
    f"{test_images:,}"
)

print(
    f"Image Size        : "
    f"{IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}"
)

print(
    "Normalization      : 0 - 1"
)

print()
print(
    f"Output Directory  : "
    f"{PREPROCESSED_ROOT.resolve()}"
)

print()
print(
    "STATUS : PREPROCESSING COMPLETED SUCCESSFULLY"
)




PREPROCESSING SUMMARY
Total Images      : 294,695
Total Classes     : 82
Train             : 87,937
Validation        : 20,564
Test              : 3,252
Image Size        : 224 x 224
Normalization      : 0 - 1

Output Directory  : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Data

STATUS : PREPROCESSING COMPLETED SUCCESSFULLY
